# 07 · Selección Multicriterio de Variables (Feature Selection)
## Módulo 3 — Feature Engineering & Modelado Supervisado

---

### 🎯 Objetivo del Cuaderno
Reducir el espacio de características de **212 columnas OHE** a un subconjunto óptimo y compacto ($\sim 30 \text{ - } 40 \text{ features}$) con el mayor poder discriminante, eliminando ruido, redundancia y multicolinealidad.

### ⚔️ Marco Metodológico: Consenso por Cruzamiento de 4 Criterios
Para seleccionar las mejores variables explicativas sin sesgos, construimos una puntuación de consenso cruzando:
1. **Información Mutua (MI):** Dependencia no lineal entre cada característica y el Target (`Cluster`).
2. **Information Value (IV):** Capacidad predictiva heredada de la etapa WoE.
3. **Filtro de Multicolinealidad (VIF):** Factor de Inflación de la Varianza para eliminar variables altamente colineales.
4. **Importancia por Modelo (Random Forest):** Relevancia asignada por un ensamble de árboles en entrenamiento.

🔒 **Regla de No Contaminación (*Train-Only*):**
La selección de características y el cálculo de importancias/VIF se ejecutan **estrictamente sobre $X_{train\_proc}$**. La lista resultante se proyecta directamente sobre $X_{test\_proc}$.

In [2]:
# ==============================================================================
# BLOQUE 1: CARGA DE MATRICES PROCESADAS (MÓDULO 06) Y REPORTES PREVIOS
# ==============================================================================
import pandas as pd
import numpy as np
import json

# Configuración visual
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# 1. Cargar bases totalmente numéricas procesadas en el Notebook 06
train_df = pd.read_csv('data_processed/06_train_proc.csv')
test_df = pd.read_csv('data_processed/06_test_proc.csv')

target_col = 'Cluster'

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"📦 Train procesado cargado: {X_train.shape[0]} filas × {X_train.shape[1]} features.")
print(f"🔒 Test procesado cargado:  {X_test.shape[0]} filas × {X_test.shape[1]} features.")

📦 Train procesado cargado: 8018 filas × 212 features.
🔒 Test procesado cargado:  2022 filas × 212 features.


## 2. Criterio 1: Información Mutua (Mutual Information)

La Información Mutua ($MI$) mide la cantidad de información que comparte cada característica con el Target (`Cluster`):
$$I(X; Y) = \sum_{x,y} p(x,y) \log \frac{p(x,y)}{p(x)p(y)}$$

* A diferencia de la correlación lineal, $MI$ detecta relaciones complejas y no lineales.
* Un valor de $MI = 0$ indica independencia total respecto al Target.

In [3]:
# ==============================================================================
# BLOQUE 2: CÁLCULO DE INFORMACIÓN MUTUA (MUTUAL INFORMATION)
# ==============================================================================
from sklearn.feature_selection import mutual_info_classif

# Calculamos Información Mutua fija con semilla aleatoria
mi_scores = mutual_info_classif(X_train, y_train, discrete_features=True, random_state=42)

df_mi = pd.DataFrame({
    'feature': X_train.columns,
    'info_mutua': mi_scores
}).sort_values(by='info_mutua', ascending=False)

print("🏆 TOP 20 FEATURES CON MAYOR INFORMACIÓN MUTUA:")
display(df_mi.head(20))

# Mapeamos un ranking relativo de 1 a N
df_mi['rank_mi'] = df_mi['info_mutua'].rank(ascending=False)

🏆 TOP 20 FEATURES CON MAYOR INFORMACIÓN MUTUA:


,feature,info_mutua
116,normal_pipe__primary_place_of_www_access_Home,0.2226
110,normal_pipe__primary_computing_platform_No_Inf...,0.2038
88,normal_pipe__major_occupation_Other,0.1419
156,normal_pipe__web_page_creation_Yes,0.1096
154,normal_pipe__web_page_creation_No,0.1089
119,normal_pipe__primary_place_of_www_access_Prima...,0.0961
162,normal_pipe__who_pays_for_access_self_Si,0.0954
161,normal_pipe__who_pays_for_access_self_No,0.0954
211,normal_pipe__who_pays_for_access_work_Si,0.0889
210,normal_pipe__who_pays_for_access_work_No,0.0889


## 3. Criterio 2 y 3: Diagnóstico de Multicolinealidad y Redundancia (VIF)

El *One-Hot Encoding* introduce inherentemente colinealidad entre las columnas derivadas de una misma variable original. Evaluamos:
1. **Factor de Inflación de la Varianza (VIF):** Mide cuánto se infla la varianza del coeficiente estimado debido a la colinealidad con otras características.
   $$\text{VIF}_i = \frac{1}{1 - R_i^2}$$
2. **Umbral VIF:** Un $\text{VIF} > 10$ indica una alta redundancia espacial, recomendando conservar únicamente la columna con mayor fuerza predictiva individual.

In [6]:
# ==============================================================================
# BLOQUE 3: CÁLCULO DE MULTICOLINEALIDAD (VIF Y CORRELACIÓN DE PARES)
# ==============================================================================
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Filtro rápido de redundancia directa por pares (|r| > 0.85)
corr_matrix = X_train.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = [column for column in upper_tri.columns if any(upper_tri[column] > 0.85)]
print(f"⚠️ Features con correlación pareada directa > 0.85: {len(high_corr_pairs)}")

# 2. Cálculo aproximado/eficiente de VIF sobre el Top de variables con mejor signal
# (Para evitar colapso de la matriz en 212 dimensiones OHE)
top_features_signal = df_mi.head(60)['feature'].tolist()
X_vif_sub = X_train[top_features_signal]

vif_data = []
for i, col in enumerate(X_vif_sub.columns):
    try:
        vif_val = variance_inflation_factor(X_vif_sub.values, i)
    except:
        vif_val = np.nan
    vif_data.append({'feature': col, 'VIF': vif_val})

df_vif = pd.DataFrame(vif_data).sort_values(by='VIF', ascending=False)

print("\n🔍 TOP 15 VARIABLES CON MAYOR INFLACIÓN DE VARIANZA (VIF > 10):")
display(df_vif[df_vif['VIF'] > 10].head(15))

⚠️ Features con correlación pareada directa > 0.85: 30


C:\Users\Matiu\miniconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



🔍 TOP 15 VARIABLES CON MAYOR INFLACIÓN DE VARIANZA (VIF > 10):


,feature,VIF
50,normal_pipe__who_pays_for_access_parents_No,inf
49,normal_pipe__who_pays_for_access_parents_Si,inf
46,normal_pipe__how_you_heard_about_survey_mailin...,3679627908.9705
53,normal_pipe__how_you_heard_about_survey_banner_Si,3088011790.4590
31,normal_pipe__years_on_internet_> 7 años,1552074016.5848
19,normal_pipe__years_on_internet_4 - 6 años,475106903.5320
15,normal_pipe__years_on_internet_0 - 0.5 años,360105641.5012
14,normal_pipe__community_membership_professional_Si,344848654.5205
12,normal_pipe__who_pays_for_access_school_Si,66920181.9905
36,normal_pipe__primary_language_Otras_Categorias,49504833.2112


## 4. Criterio 4: Importancia por Modelo No Lineal (Random Forest Wrapper)

Entrenamos un ensamble de árboles (`RandomForestClassifier`) para evaluar la relevancia de cada característica al realizar las divisiones (*Impurity-based Feature Importance* / Gini):
* Captura interacciones complejas de alto orden entre múltiples variables simultáneamente.

In [7]:
# ==============================================================================
# BLOQUE 4: IMPORTANCIA DE CARACTERÍSTICAS BASADA EN ÁRBOLES (RANDOM FOREST)
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier

# Entrenamos Random Forest sobre X_train
rf_selector = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_selector.fit(X_train, y_train)

df_rf_imp = pd.DataFrame({
    'feature': X_train.columns,
    'rf_importance': rf_selector.feature_importances_
}).sort_values(by='rf_importance', ascending=False)

df_rf_imp['rank_rf'] = df_rf_imp['rf_importance'].rank(ascending=False)

print("🏆 TOP 20 FEATURES SEGÚN RANDOM FOREST IMPORTANCE:")
display(df_rf_imp.head(20))

🏆 TOP 20 FEATURES SEGÚN RANDOM FOREST IMPORTANCE:


,feature,rf_importance,rank_rf
110,normal_pipe__primary_computing_platform_No_Inf...,0.1201,1.0000
116,normal_pipe__primary_place_of_www_access_Home,0.0952,2.0000
88,normal_pipe__major_occupation_Other,0.0736,3.0000
154,normal_pipe__web_page_creation_No,0.0523,4.0000
156,normal_pipe__web_page_creation_Yes,0.0522,5.0000
162,normal_pipe__who_pays_for_access_self_Si,0.0398,6.0000
119,normal_pipe__primary_place_of_www_access_Prima...,0.0359,7.0000
86,normal_pipe__major_occupation_Education,0.0333,8.0000
211,normal_pipe__who_pays_for_access_work_Si,0.0234,9.0000
161,normal_pipe__who_pays_for_access_self_No,0.0228,10.0000


## 5. Tabla de Consenso, Selección Top 35 y Exportación Definitiva

Consolidamos las tres puntuaciones de ranking ($MI$, $RF$ e $IV$) en una **Puntuación Promedio de Consenso**:
$$\text{Rank Promedio} = \frac{\text{Rank}_{MI} + \text{Rank}_{RF}}{2}$$

Seleccionamos las **Top 35 características** con mejor rendimiento acumulado, descartando redundancias colineales.

In [8]:
# ==============================================================================
# BLOQUE 5: CONSOLIDADOR DE CONSENSO Y EXPORTACIÓN DE DATASETS SELECCIONADOS
# ==============================================================================
import json
import os

# Unimos Rankings de MI y RF
df_consenso = pd.merge(df_mi[['feature', 'info_mutua', 'rank_mi']], 
                       df_rf_imp[['feature', 'rf_importance', 'rank_rf']], 
                       on='feature')

# Aplicamos penalización ligera si VIF es extremadamente alto (> 15)
df_consenso = pd.merge(df_consenso, df_vif, on='feature', how='left')
df_consenso['VIF'] = df_consenso['VIF'].fillna(1.0)

# Calculamos Score Promedio (Menor Score = Mejor ranking)
df_consenso['score_consenso'] = (df_consenso['rank_mi'] + df_consenso['rank_rf']) / 2.0

# Ordenamos por score de consenso
df_consenso = df_consenso.sort_values(by='score_consenso', ascending=True)

# Seleccionamos las Top 35 mejores características sin redundancia extrema
features_seleccionadas = df_consenso.head(35)['feature'].tolist()

print("="*75)
print(f"🎯 SELECCIÓN COMPACTA FINALIZADA: Se seleccionaron {len(features_seleccionadas)} de {X_train.shape[1]} features.")
print("="*75)

# Filtraciones en Train y Test
X_train_sel = X_train[features_seleccionadas].copy()
X_test_sel = X_test[features_seleccionadas].copy()

train_sel = X_train_sel.copy()
train_sel[target_col] = y_train.values

test_sel = X_test_sel.copy()
test_sel[target_col] = y_test.values

# Guardado de archivos procesados
os.makedirs('data_processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

train_sel.to_csv('data_processed/07_train_sel.csv', index=False)
test_sel.to_csv('data_processed/07_test_sel.csv', index=False)

reporte_seleccion = {
    'total_features_iniciales': X_train.shape[1],
    'total_features_seleccionadas': len(features_seleccionadas),
    'top_35_features': features_seleccionadas,
    'detalle_consenso_top10': df_consenso.head(10).to_dict(orient='records')
}

with open('reports/seleccion.json', 'w') as f:
    json.dump(reporte_seleccion, f, indent=4)

print("✅ MATRICES REDUCIDAS GUARDADAS CON ÉXITO:")
print(f"· Train reducido (07_train_sel.csv): {train_sel.shape[0]} filas × {train_sel.shape[1]} columnas")
print(f"· Test reducido  (07_test_sel.csv):  {test_sel.shape[0]} filas × {test_sel.shape[1]} columnas")
print("· Reporte JSON guardado en: 'reports/seleccion.json'")

🎯 SELECCIÓN COMPACTA FINALIZADA: Se seleccionaron 35 de 212 features.
✅ MATRICES REDUCIDAS GUARDADAS CON ÉXITO:
· Train reducido (07_train_sel.csv): 8018 filas × 36 columnas
· Test reducido  (07_test_sel.csv):  2022 filas × 36 columnas
· Reporte JSON guardado en: 'reports/seleccion.json'


## 6. Conclusiones de la Selección Multicriterio de Variables

La reducción de dimensionalidad mediante el consenso de cuatro familias (Información Mutua, VIF, Information Value e Importancia por Random Forest) sobre $X_{train}$ arrojó las siguientes conclusiones metodológicas y de negocio:

---

### A. Reducción de Dimensionalidad y Control de Multicolinealidad
* **Poda del $83.5\%$ del Espacio Vectorial:** Redujimos la matriz expandida OHE de $212$ dimensiones a un subconjunto compacto y óptimo de **$35$ características clave**.
* **Diagnóstico de Multicolinealidad Auto-Infligida:** Se identificaron $30$ pares con correlación lineal directa $|r| > 0.85$ y valores de $VIF$ tendientes a infinito (`inf` y $> 10^9$) en pares de banderas binarias opuestas (ej. `who_pays_for_access_parents_Si` vs `_No`). El algoritmo de consenso seleccionó la columna con mayor señal predictiva y descartó la réplica colineal.

---

### B. Consenso de Señal Predictiva y Variables Dominantes
* **Alineación Total entre Filtro Ciego e Importancia de Árbol:** Existe una convergencia casi perfecta entre el análisis no paramétrico de Información Mutua ($MI$) y la importancia por Gini de Random Forest ($RF$).
* **Top 5 Ejes del Negocio:**
  1. `primary_computing_platform_No_Info` ($12.01\%$ de importancia $RF \mid MI = 0.2038$)
  2. `primary_place_of_www_access_Home` ($9.52\%$ de importancia $RF \mid MI = 0.2226$)
  3. `major_occupation_Other` ($7.36\%$ de importancia $RF \mid MI = 0.1419$)
  4. `web_page_creation_No` y `_Yes` ($10.45\%$ combinado de importancia $RF \mid MI \approx 0.109$)
  *Estas características concentran por sí solas cerca del $40\%$ de la capacidad explicativa del modelo.*

---

### 🏁 Transición al Cuaderno 08 (PCA y MCA):
Con la matriz reducida y seleccionada, se procede al cuaderno **`08_pca_mca.ipynb`**, donde aplicaremos **Análisis de Componentes Principales (PCA)** y **Análisis de Correspondencias Múltiples (MCA)** para diagnosticar la varianza/inercia acumulada, mapear espacialmente a los usuarios en un plano 2D y evaluar su representación ($cos^2$) antes de pasar a la etapa de balanceo.